# RAGAS Evaluation — ERP Procurement Intelligence

Evaluates the RAG pipeline quality across 4 metrics:

| Metric | What it measures |
|--------|------------------|
| **Faithfulness** | Is the answer grounded in the retrieved context? |
| **Answer Relevancy** | Does the answer address the question? |
| **Context Precision** | Are retrieved chunks actually relevant? |
| **Context Recall** | Do the chunks contain what's needed to answer? |

Only RAG and Hybrid queries are evaluated (SQL queries have no retrieval step).

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dotenv import load_dotenv
load_dotenv('../.env')

print('Setup complete')

## 1. Run Smoke Test
Verify all agents are working before the expensive RAGAS run.

In [ ]:
from src.evaluation.eval_runner import run_smoke_test
all_ok = run_smoke_test()

## 2. Collect Evaluation Samples
Run each test question through the pipeline and capture (question, answer, contexts, ground_truth).

In [ ]:
from src.evaluation.ragas_eval import (
    load_test_questions,
    collect_eval_samples,
)
from src.agents.rag_agent import RAGAgent
from src.graph.langgraph_workflow import build_workflow
from src.vectorstore.chroma_store import ChromaStore

store     = ChromaStore()
rag_agent = RAGAgent(store=store)
workflow  = build_workflow(rag_agent=rag_agent, chroma_store=store)

test_questions = load_test_questions('../data/sample_queries.json')
print(f'Loaded {len(test_questions)} test questions')

# Show test questions breakdown by route
routes = {}
for q in test_questions:
    r = q.get('route', 'unknown')
    routes[r] = routes.get(r, 0) + 1
print('Route breakdown:', routes)

In [ ]:
# Collect samples - this calls Bedrock for each question (~2-5 mins)
samples = collect_eval_samples(
    test_questions = test_questions,
    workflow       = workflow,
    rag_agent      = rag_agent,
)
print(f'\nCollected {len(samples)} samples')
print(f'Successful: {sum(1 for s in samples if s.success)}')
print(f'Failed:     {sum(1 for s in samples if not s.success)}')

In [ ]:
# Preview sample data
df_samples = pd.DataFrame([{
    'route'      : s.route,
    'question'   : s.question[:60] + '...',
    'answer'     : s.answer[:80] + '...',
    'n_contexts' : len(s.contexts),
    'latency_ms' : s.latency_ms,
    'success'    : s.success,
} for s in samples])

df_samples

## 3. Run RAGAS Evaluation

In [ ]:
from src.evaluation.ragas_eval import run_ragas

# RAGAS runs on RAG+hybrid samples only
scores = run_ragas(samples)
print('\nRAGAS Scores:')
for k, v in scores.items():
    print(f'  {k:<25}: {v:.4f}')

## 4. Results Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RAGAS Evaluation — ERP Procurement Intelligence', fontsize=14, fontweight='bold')

# --- Bar chart: metric scores ---
ax1     = axes[0]
metrics = list(scores.keys())
values  = list(scores.values())
colors  = ['#28a745' if v >= 0.80 else ('#ffc107' if v >= 0.65 else '#dc3545') for v in values]

bars = ax1.barh(metrics, values, color=colors, height=0.5, edgecolor='white')
ax1.set_xlim(0, 1.0)
ax1.axvline(x=0.80, color='gray', linestyle='--', alpha=0.7, label='Target (0.80)')
ax1.set_xlabel('Score')
ax1.set_title('Metric Scores')
ax1.legend()

for bar, val in zip(bars, values):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

# --- Pie chart: route distribution ---
ax2 = axes[1]
route_counts = {}
for s in samples:
    route_counts[s.route] = route_counts.get(s.route, 0) + 1

route_colors = {'sql': '#1a6de0', 'rag': '#28a745', 'hybrid': '#7b2dbd'}
ax2.pie(
    route_counts.values(),
    labels   = [f"{k.upper()} ({v})" for k, v in route_counts.items()],
    colors   = [route_colors.get(k, '#999') for k in route_counts],
    autopct  = '%1.0f%%',
    startangle=90,
)
ax2.set_title('Evaluation Samples by Route')

plt.tight_layout()
plt.savefig('../data/ragas_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to data/ragas_results.png')

In [ ]:
# Latency analysis
df_latency = pd.DataFrame([{
    'route'      : s.route,
    'latency_ms' : s.latency_ms,
} for s in samples if s.success])

print('Latency by route (ms):')
print(df_latency.groupby('route')['latency_ms'].agg(['mean', 'min', 'max']).round(0))

## 5. Save Results

In [ ]:
from datetime import datetime, timezone
from src.evaluation.ragas_eval import EvalResults

avg_latency = sum(s.latency_ms for s in samples if s.success) / max(sum(1 for s in samples if s.success), 1)

results = EvalResults(
    faithfulness       = scores['faithfulness'],
    answer_relevancy   = scores['answer_relevancy'],
    context_precision  = scores['context_precision'],
    context_recall     = scores['context_recall'],
    n_samples          = len(samples),
    n_rag_samples      = sum(1 for s in samples if s.route in ('rag','hybrid')),
    avg_latency_ms     = round(avg_latency, 1),
    timestamp          = datetime.now(timezone.utc).isoformat(),
    samples            = samples,
)

with open('../data/evaluation_results.json', 'w') as f:
    json.dump(results.to_dict(), f, indent=2)

print(results.summary())